# Chapter 11: Segmentation done right

In [1]:
import numpy as np
import pandas as pd
from expkit.segments.behavioral import simulate_population, BEHAVIORAL_LABELS
from expkit.plot.style import apply_style
apply_style()

## Population breakdown

In [2]:
df = simulate_population(5000, seed=110)
print(df['segment'].value_counts())
print('axis means by segment:')
print(df.groupby('segment')[['weekly_active_rate', 'contribution_rate', 'intentional_rate']].mean().round(3))

segment
passive_consumer      1898
silent_intentional    1249
active_contributor    1228
active_consumer        625
Name: count, dtype: int64
axis means by segment:
                    weekly_active_rate  contribution_rate  intentional_rate
segment                                                                    
active_consumer                  0.414              0.072             0.474
active_contributor               0.409              0.241             0.334
passive_consumer                 0.245              0.127             0.189
silent_intentional               0.159              0.154             0.476


### Why the buckets are unbalanced

The labels come from a median-split cascade in `expkit.segments.behavioral.label_behavioral`. Medians on three independent axes do not produce four equal buckets. Default is `passive_consumer`. Override to `active_contributor` when both `high_active` and `high_contrib`. Else override to `active_consumer` when `high_active` and not `high_contrib` and `high_intent`. Else override to `silent_intentional` when not `high_active` and `high_intent`. The smallest bucket (`active_consumer`) requires the joint condition `high_active` and not `high_contrib` and `high_intent`, which is a small slice of the joint distribution. Keep these counts in mind when reading Loop D below: the partial-pooling effect is largest exactly for the smallest bucket.

## Demographic vs behavioural slicing

In [3]:
rng = np.random.default_rng(110)
n = 5000
df = simulate_population(n, seed=110)
df['country'] = rng.choice(['A', 'B', 'C'], size=n)
df['arm'] = rng.choice(['control', 'treatment'], size=n)
treat_lift = {'active_contributor': 0.10, 'active_consumer': 0.04, 'silent_intentional': -0.03, 'passive_consumer': 0.0}
df['outcome'] = 0
base = 0.30
for seg, l in treat_lift.items():
    for arm in ['control', 'treatment']:
        m = (df['segment'] == seg) & (df['arm'] == arm)
        p = base + (l if arm == 'treatment' else 0.0)
        df.loc[m, 'outcome'] = rng.binomial(1, max(0, min(1, p)), size=int(m.sum()))
print('--- by country ---')
print(df.groupby(['country', 'arm'])['outcome'].mean().unstack().round(3))
print('\n--- by behavioural segment ---')
print(df.groupby(['segment', 'arm'])['outcome'].mean().unstack().round(3))
print('\nPooled treatment - control =', round(df.groupby('arm')['outcome'].mean().diff().iloc[-1], 3))

--- by country ---
arm      control  treatment
country                    
A          0.315      0.343
B          0.296      0.352
C          0.316      0.329

--- by behavioural segment ---
arm                 control  treatment
segment                               
active_consumer       0.261      0.390
active_contributor    0.354      0.420
passive_consumer      0.299      0.329
silent_intentional    0.302      0.262

Pooled treatment - control = 0.033


## Loop D: recovering simulator-injected lifts and quantifying shrinkage

Loop D is a recovery exercise: the simulator bakes in lifts of +0.10, +0.04, -0.03, 0.0 by segment, and we ask the hierarchical model to recover those known values. The cell below loads the saved posterior from `data/hierarchical_posterior.nc`, prints the posterior of `tau` (population scale of segment effects on the logit scale) and `mu` (population mean), and computes the shrinkage of each segment's posterior mean toward `mu`, relative to the no-pooling independent estimate. Shrinkage near 0 means the segment dominates its own posterior. Shrinkage near 1 means the posterior collapses to the population mean.

In [ ]:
from pathlib import Path
import arviz as az

# Load the saved trace produced by generate.py (run that script first if missing).
posterior_path = Path('data') / 'hierarchical_posterior.nc'
if posterior_path.exists():
    idata = az.from_netcdf(posterior_path)
    mu_post = float(idata.posterior['mu'].mean().values)
    tau_samples = idata.posterior['tau'].values.ravel()
    tau_mean = float(tau_samples.mean())
    tau_lo, tau_hi = (float(np.quantile(tau_samples, 0.025)), float(np.quantile(tau_samples, 0.975)))
    print(f'tau posterior mean = {tau_mean:.3f}, 95% CI = [{tau_lo:.3f}, {tau_hi:.3f}] (logit scale)')
    print(f'mu  posterior mean = {mu_post:.3f} (logit scale)')

    # Recover the same successes/n table generate.py used (seed=115, n=4000)
    rng = np.random.default_rng(115)
    n_total = 4000
    df_h = simulate_population(n_total, seed=115)
    df_h['arm'] = rng.choice(['control', 'treatment'], size=n_total)
    treat_lift = {'active_contributor': 0.10, 'active_consumer': 0.04, 'silent_intentional': -0.03, 'passive_consumer': 0.0}
    df_h['outcome'] = 0
    base = 0.30
    for seg, l in treat_lift.items():
        for arm in ['control', 'treatment']:
            m = (df_h['segment'] == seg) & (df_h['arm'] == arm)
            p = base + (l if arm == 'treatment' else 0.0)
            df_h.loc[m, 'outcome'] = rng.binomial(1, max(0, min(1, p)), size=int(m.sum()))

    segments = list(idata.posterior.coords['segment'].values)
    rows = []
    for i, seg in enumerate(segments):
        sub = df_h[df_h['segment'] == seg]
        s_t = int(sub[sub['arm'] == 'treatment']['outcome'].sum())
        n_t = int((sub['arm'] == 'treatment').sum())
        s_c = int(sub[sub['arm'] == 'control']['outcome'].sum())
        n_c = int((sub['arm'] == 'control').sum())
        ind_prob = (s_t / n_t) - (s_c / n_c)
        # logit-scale independent estimate of the treatment effect
        from math import log
        logit_t = log((s_t / n_t) / (1 - s_t / n_t))
        logit_c = log((s_c / n_c) / (1 - s_c / n_c))
        ind_logit = logit_t - logit_c
        # Posterior mean of the per-segment effect on the logit scale
        eff = idata.posterior['effect'].sel(segment=seg).values.ravel()
        post_logit = float(eff.mean())
        denom = ind_logit - mu_post
        shrinkage = (ind_logit - post_logit) / denom if abs(denom) > 1e-9 else float('nan')
        rows.append({
            'segment': seg,
            'n_total': n_t + n_c,
            'injected_lift_pp': treat_lift[seg],
            'independent_lift_pp': round(ind_prob, 3),
            'independent_logit': round(ind_logit, 3),
            'posterior_mean_logit': round(post_logit, 3),
            'shrinkage': round(shrinkage, 3),
        })
    print()
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print(f'No posterior file at {posterior_path}. Run generate.py to produce it.')